# Control impact dashboard

Load runs (or an experiment), compute outcomes and control impact, and plot:
- Scatter: control vs metric
- Arm bar charts with CI
- Parallel coordinates: control profile vs objective vector

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import os
os.environ["RAPBOT_USE_DB"] = "1"

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load run outcomes (set experiment_id or use recent runs)
from evo_rhyme import db
from scripts.analyze_control_impact import load_run_outcomes

EXPERIMENT_ID = None  # set to int to filter by experiment
LIMIT = 80

if EXPERIMENT_ID:
    run_list = db.list_runs_for_experiment(EXPERIMENT_ID, limit=LIMIT)
    run_ids = [r["run_id"] for r in run_list]
else:
    run_list = db.list_runs(limit=LIMIT)
    run_ids = [r["run_id"] for r in run_list]

rows = load_run_outcomes(run_ids, aggregation_mode="best", top_k=5)
print(f"Loaded {len(rows)} runs")

In [ ]:
# Control impact report
from evo_rhyme.experiment_analysis import analyze_control_impact

report = analyze_control_impact(rows, bootstrap_n=300)
print("Runs:", report.get("runs"))
print("By control:", list(report.get("by_control", {}).keys()))

In [ ]:
# Scatter: numeric control vs fitness
df = pd.DataFrame([{"run_id": r["run_id"], "fitness": r.get("fitness") or 0, **{k: v for k, v in (r.get("controls") or {}).items() if isinstance(v, (int, float))}} for r in rows])
numeric_cols = [c for c in df.columns if c != "run_id" and c != "fitness" and df[c].dtype in ("int64", "float64")]
if numeric_cols:
    fig, axes = plt.subplots(1, min(3, len(numeric_cols)), figsize=(4 * min(3, len(numeric_cols)), 4))
    if len(numeric_cols) == 1:
        axes = [axes]
    for i, col in enumerate(numeric_cols[:3]):
        axes[i].scatter(df[col], df["fitness"], alpha=0.6)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel("fitness")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric controls in data.")

In [ ]:
# Bar chart: mean fitness by categorical control (first varying control)
by_control = report.get("by_control") or {}
for ck, data in list(by_control.items())[:2]:
    ci_by = data.get("ci_by_value") or {}
    if not ci_by:
        continue
    vals = list(ci_by.keys())
    means = [ci_by[v]["mean"] for v in vals]
    lo = [ci_by[v]["ci_low"] for v in vals]
    hi = [ci_by[v]["ci_high"] for v in vals]
    plt.figure(figsize=(8, 4))
    plt.bar(range(len(vals)), means, yerr=[[means[i] - lo[i] for i in range(len(vals))], [hi[i] - means[i] for i in range(len(vals))]], capsize=4)
    plt.xticks(range(len(vals)), [str(v)[:12] for v in vals], rotation=45, ha="right")
    plt.ylabel("fitness (mean ± 95% CI)")
    plt.title(f"Control: {ck}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Parallel coordinates: control profile vs objective vector (sample)
from evo_rhyme.experiment_metrics import FITNESS_VECTOR_KEYS

flat = []
for r in rows[:50]:
    c = r.get("controls") or {}
    v = r.get("fitness_vector") or {}
    flat.append({
        "fitness": r.get("fitness") or 0,
        **{f"ctrl_{k}": (c.get(k) if isinstance(c.get(k), (int, float)) else hash(str(c.get(k))) % 1000) for k in list(c)[:5]},
        **{f"out_{k}": v.get(k, 0) for k in FITNESS_VECTOR_KEYS if k in v}
    })
df2 = pd.DataFrame(flat)
if len(df2.columns) >= 2:
    cols = ["fitness"] + [k for k in FITNESS_VECTOR_KEYS if k in df2.columns]
    df2[cols].plot(alpha=0.4, figsize=(10, 4))
    plt.xlabel("run index")
    plt.ylabel("score")
    plt.title("Fitness and objective axes (sample)")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough dimensions.")